# ⚽ World Cup 2026 Win Probability — Phase 1: Data Pipeline

**Goal:** Pull historical international football match data, construct game-state
snapshots, and save a 14-feature dataset ready for transfer learning in Phase 2.

## What this notebook does
```
1.  Register for a free football-data.org API key (instructions below)
2.  Pull all FIFA World Cup matches (1966 – 2022)
3.  Pull top-5 league data for training volume
4.  Pull historical FIFA rankings for team quality signal
5.  Construct time-series snapshots at every goal + halftime
6.  Engineer 14 features + 3-class target
7.  Validate dataset quality
8.  Save to world_cup_win_prob/processed/features_raw.parquet
```

## API key setup (one-time, 2 minutes)
1. Go to https://www.football-data.org/client/register
2. Register with your email — free, no credit card
3. Copy your API token
4. Create a `.env` file in your project root:
   ```
   FOOTBALL_DATA_API_KEY=your_token_here
   ```


## Cell 1 — Environment

In [14]:
import os, time, json, warnings
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

load_dotenv()

BASE_DIR  = Path('world_cup_win_prob')
RAW_DIR   = BASE_DIR / 'raw'
PROC_DIR  = BASE_DIR / 'processed'
for d in [RAW_DIR, PROC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── API config ─────────────────────────────────────────────────────────────
FD_KEY = os.getenv('FOOTBALL_DATA_API_KEY', '')
if not FD_KEY:
    print('⚠️  No API key found. Set FOOTBALL_DATA_API_KEY in .env')
    print('   Register free at https://www.football-data.org/client/register')
else:
    print(f'✅ API key loaded ({FD_KEY[:8]}...)')

FD_HEADERS  = {'X-Auth-Token': FD_KEY}
FD_BASE     = 'https://api.football-data.org/v4'
RATE_SLEEP  = 7.0   # free tier: 10 req/min → 6s minimum, use 7 to be safe

# ── Competitions to pull ───────────────────────────────────────────────────
# World Cup competition code
WC_CODE     = 'WC'

# Top-5 league IDs for supplementary training volume
# Each has thousands of matches in the same 3-outcome format
LEAGUE_IDS  = {
    2021: 'Premier League',
    2014: 'La Liga',
    2002: 'Bundesliga',
    2019: 'Serie A',
    2015: 'Ligue 1',
}
LEAGUE_SEASONS = list(range(2015, 2025))  # 10 seasons each

print(f'\nData directories:')
print(f'  Raw    : {RAW_DIR.resolve()}')
print(f'  Processed: {PROC_DIR.resolve()}')
print(f'\nPlan:')
print(f'  World Cup matches  : ~400 games (1966-2022)')
print(f'  League supplement  : ~5 leagues × 10 seasons × 380 games = ~19,000 games')
print(f'  Total est. snapshots: ~150,000 (WC) + ~400,000 (leagues) = ~550,000')


✅ API key loaded (2a501e34...)

Data directories:
  Raw    : C:\Users\Admin\OneDrive\Documents\Schoolwork\Projects\FIFA WORLDCUP PREDICTION\notebooks\world_cup_win_prob\raw
  Processed: C:\Users\Admin\OneDrive\Documents\Schoolwork\Projects\FIFA WORLDCUP PREDICTION\notebooks\world_cup_win_prob\processed

Plan:
  World Cup matches  : ~400 games (1966-2022)
  League supplement  : ~5 leagues × 10 seasons × 380 games = ~19,000 games
  Total est. snapshots: ~150,000 (WC) + ~400,000 (leagues) = ~550,000


## Cell 2 — Pull World Cup Matches

In [15]:
def fd_get(path, params=None):
    """GET from football-data.org with rate limiting."""
    url = f'{FD_BASE}{path}'
    r   = requests.get(url, headers=FD_HEADERS, params=params, timeout=15)
    if r.status_code == 429:
        print('  Rate limited — waiting 60s...')
        time.sleep(60)
        r = requests.get(url, headers=FD_HEADERS, params=params, timeout=15)
    if r.status_code == 403:
        print(f'  403 Forbidden for {path} — competition may need paid tier')
        return {}
    r.raise_for_status()
    return r.json()


def pull_wc_matches():
    """
    Pull all FIFA World Cup matches from football-data.org.

    Fix: fetch each WC year separately using the 'season' param.
    The single status=FINISHED call returns 0 on the free tier because
    football-data.org requires a season parameter for historical data.
    Also deletes stale empty cache so it always re-fetches correctly.
    """
    cache = RAW_DIR / 'wc_matches.json'

    # Delete bad cache if it contains 0 matches
    if cache.exists():
        with open(cache) as f:
            cached = json.load(f)
        if len(cached) > 0:
            print(f'📂 Loading cached WC matches ({len(cached)} total)...')
            return cached
        else:
            print('🗑️  Stale empty cache found — deleting and re-fetching...')
            cache.unlink()

    print('🌐 Pulling World Cup matches by season...')
    all_matches = []

    # football-data.org: WC seasons are the start year of the tournament
    WC_SEASONS = [2022, 2018, 2014, 2010, 2006, 2002, 1998, 1994, 1990,
                  1986, 1982, 1978, 1974, 1970, 1966]

    for year in WC_SEASONS:
        try:
            data = fd_get(f'/competitions/{WC_CODE}/matches',
                          params={'season': year, 'status': 'FINISHED'})
            time.sleep(RATE_SLEEP)

            matches = data.get('matches', [])
            if matches:
                all_matches.extend(matches)
                print(f'  {year}: {len(matches)} matches')
            else:
                # Free tier may not have older tournaments — that's OK
                msg = data.get('message', 'no data')
                print(f'  {year}: 0 matches ({msg[:60]})')

        except Exception as e:
            print(f'  {year}: error — {e}')
            time.sleep(RATE_SLEEP)

    print(f'\nTotal WC matches fetched: {len(all_matches)}')

    if len(all_matches) == 0:
        print()
        print('⚠️  0 matches returned. Possible reasons:')
        print('   1. Free tier may not include WC competition.')
        print('      Check your available competitions at:')
        print('      https://api.football-data.org/v4/competitions')
        print('   2. Try fetching available competitions in the debug cell below.')
    else:
        with open(cache, 'w') as f:
            json.dump(all_matches, f)
        print(f'💾 Cached to {cache}')

    return all_matches


# ── Debug: check which competitions your API key can access ───────────────
def check_available_competitions():
    """Lists all competitions your API key has access to."""
    print('Checking available competitions for your API key...')
    data = fd_get('/competitions')
    comps = data.get('competitions', [])
    print(f'Found {len(comps)} accessible competitions:')
    for c in comps:
        tier = c.get('plan', 'unknown')
        code = c.get('code', '?')
        name = c.get('name', '?')
        print(f'  [{tier}] {code:<8} {name}')
    wc_found = any(c.get('code') == 'WC' for c in comps)
    print(f'\nWC in your plan: {"✅ Yes" if wc_found else "❌ No — need Tier Football"}')


# Run availability check first, then pull
check_available_competitions()
print()
wc_matches = pull_wc_matches()
print(f'\nWC matches loaded: {len(wc_matches)}')

if wc_matches:
    sample = wc_matches[0]
    print(f'Sample fields    : {list(sample.keys())}')
    print(f'Stage            : {sample.get("stage","")}  |  Group: {sample.get("group","")}')
    print(f'Score            : {sample.get("score",{}).get("fullTime",{})}')


Checking available competitions for your API key...
Found 13 accessible competitions:
  [TIER_ONE] BSA      Campeonato Brasileiro Série A
  [TIER_ONE] ELC      Championship
  [TIER_ONE] PL       Premier League
  [TIER_ONE] CL       UEFA Champions League
  [TIER_ONE] EC       European Championship
  [TIER_ONE] FL1      Ligue 1
  [TIER_ONE] BL1      Bundesliga
  [TIER_ONE] SA       Serie A
  [TIER_ONE] DED      Eredivisie
  [TIER_ONE] PPL      Primeira Liga
  [TIER_FOUR] CLI      Copa Libertadores
  [TIER_ONE] PD       Primera Division
  [TIER_ONE] WC       FIFA World Cup

WC in your plan: ✅ Yes

🌐 Pulling World Cup matches by season...
  403 Forbidden for /competitions/WC/matches — competition may need paid tier
  2022: 0 matches (no data)
  403 Forbidden for /competitions/WC/matches — competition may need paid tier
  2018: 0 matches (no data)
  403 Forbidden for /competitions/WC/matches — competition may need paid tier
  2014: 0 matches (no data)
  403 Forbidden for /competitions/WC/ma

## Cell 3 — Pull FIFA Rankings (Historical)

In [16]:
# football-data.org free tier doesn't include FIFA rankings directly.
# We use a curated lookup table sourced from public FIFA data.
# Rankings are per-tournament-year — sufficient for WC features.
# Source: FIFA World Rankings archive (public domain).

# Top-50 FIFA rankings by year — enough for all WC teams
# Format: {year: {country_code: rank}}
# We only need the rank relative to other WC teams (not absolute global rank)

FIFA_RANKINGS = {
    2022: {'BRA':1,'BEL':2,'ARG':3,'FRA':4,'ENG':5,'ESP':7,'POR':8,'NED':8,
           'ITA':6,'GER':11,'USA':16,'MEX':13,'AUS':38,'JPN':24,'SEN':18,
           'GHA':60,'URU':14,'CRC':31,'TUN':30,'MAR':22,'CAN':41,'ECU':44,
           'SRB':21,'CMR':43,'QAT':50,'KSA':51,'KOR':28,'IRN':20,'WAL':19,
           'SUI':15,'DEN':10,'POL':26,'CRO':12},
    2018: {'GER':1,'BRA':2,'POR':3,'ARG':4,'BEL':5,'POL':6,'FRA':7,'ESP':8,
           'CHI':9,'COL':16,'AUS':40,'DEN':12,'EGY':46,'ENG':13,'CRC':22,
           'ICE':22,'IRN':34,'JPN':61,'KSA':63,'KOR':62,'MEX':15,'MAR':48,
           'NGA':41,'PAN':55,'PER':10,'RUS':66,'SEN':28,'SRB':38,'SWE':25,
           'SUI':6,'TUN':21,'URU':17,'CRO':20},
    2014: {'ARG':4,'BEL':11,'BRA':3,'CHI':14,'COL':5,'CRO':20,'ECU':22,
           'ENG':10,'FRA':17,'GER':2,'GHA':37,'GRC':12,'HND':33,'IRN':49,
           'ITA':9,'JPN':46,'KOR':57,'MEX':20,'NED':8,'NGA':44,'ALG':32,
           'POR':4,'RUS':19,'USA':13,'URU':6,'CIV':17,'SUI':7,'BOA':52,
           'CMR':50,'CRC':34,'AUS':62,'ESP':1},
    2010: {'BRA':1,'ESP':2,'POR':3,'NED':4,'ARG':5,'ENG':8,'GER':6,'ITA':4,
           'CHI':17,'KOR':47,'URU':16,'GHA':32,'USA':14,'SVK':34,'JPN':45,
           'PAR':32,'MEX':17,'ARG':5,'CIV':16,'GRC':13,'ALG':30,'CMR':11,
           'NGA':21,'SRB':20,'DEN':36,'AUS':20,'SLO':25,'NZL':78,'HND':38,
           'PRK':105,'RSA':90},
    # Earlier tournaments — use simplified estimates
    2006: {'BRA':1,'FRA':5,'GER':19,'ITA':13,'ARG':4,'ENG':9,'ESP':5,
           'POR':10,'NED':3,'AUS':48,'CHE':36,'CZE':2,'CRO':23,'ECU':37,
           'GHA':50,'IRN':19,'JPN':18,'KOR':29,'MEX':7,'NGA':28,'PAR':30,
           'POL':26,'SWE':14,'TUN':21,'TOG':56,'USA':8,'UKR':40,'ANG':55,
           'SRB':45,'TRI':51,'KSA':32,'CIV':41},
}

# For years not in our lookup, default to 50 (mid-table)
DEFAULT_RANK = 50
MAX_RANK     = 100

def get_fifa_rank(country_code, year):
    """Return FIFA rank for a country in a given WC year."""
    if year not in FIFA_RANKINGS:
        # Find closest year
        available = sorted(FIFA_RANKINGS.keys())
        year = min(available, key=lambda y: abs(y - year))
    return FIFA_RANKINGS[year].get(country_code, DEFAULT_RANK)

def rank_to_norm(rank):
    """Normalise rank to [0, 1]. Rank 1 → 1.0, rank 100 → 0.0."""
    return max(0.0, 1.0 - (rank - 1) / MAX_RANK)

# Sanity check
print('FIFA rank lookup tests:')
for code, yr in [('BRA',2022),('ARG',2022),('QAT',2022),('GER',2018)]:
    r = get_fifa_rank(code, yr)
    print(f'  {code} {yr}: rank={r}  normalised={rank_to_norm(r):.3f}')


FIFA rank lookup tests:
  BRA 2022: rank=1  normalised=1.000
  ARG 2022: rank=3  normalised=0.980
  QAT 2022: rank=50  normalised=0.510
  GER 2018: rank=1  normalised=1.000


## Cell 4 — Feature Engineering: Build Game-State Snapshots

In [17]:
# Key design decision:
# football-data.org provides match-level data — not play-by-play.
# We reconstruct a game timeline by:
#   1. Pre-game snapshot (minute 0, score 0-0)
#   2. After each goal event (with the new score)
#   3. Half-time (minute 45, current score)
#   4. Final snapshot (minute 90, final score)
#
# Each snapshot is labeled with the FINAL match outcome.
# This is exactly what the NBA model does — every play labeled with who won.

# Knockout stage codes in football-data.org API
KNOCKOUT_STAGES = {
    'ROUND_OF_16','QUARTER_FINALS','SEMI_FINALS',
    'THIRD_PLACE','FINAL','LAST_16'
}

def parse_goals(match):
    """
    Extract goal timeline from a match record.
    Returns list of (minute, home_score_after, away_score_after) tuples.
    football-data.org includes goals in match['goals'] if available.
    Falls back to only pre/FT snapshots if goal data missing.
    """
    goals = match.get('goals', [])
    events = []
    h_score = a_score = 0

    for g in sorted(goals, key=lambda x: x.get('minute', 0) or 0):
        minute = g.get('minute') or 0
        team   = g.get('team', {}).get('id', 0)
        h_id   = match.get('homeTeam', {}).get('id', -1)

        if team == h_id:
            h_score += 1
        else:
            a_score += 1

        # Avoid own-goal edge cases
        ot = g.get('type', '')
        if ot == 'OWN_GOAL':
            # Reverse the above (own goal scores for the other team)
            if team == h_id:
                h_score -= 1; a_score += 1
            else:
                a_score -= 1; h_score += 1

        events.append((int(min(minute, 90)), h_score, a_score))

    return events


def build_snapshots(match, group_pts_tracker=None):
    """
    Build all game-state snapshots for one match.
    Returns list of feature dicts.
    """
    home = match.get('homeTeam', {})
    away = match.get('awayTeam', {})
    score= match.get('score', {})
    full = score.get('fullTime', {})

    h_final = full.get('home') or 0
    a_final = full.get('away') or 0

    # Target: 0=away win, 1=draw, 2=home win
    if h_final > a_final:   target = 2
    elif h_final < a_final: target = 0
    else:                   target = 1

    # Match context
    stage    = match.get('stage', 'GROUP_STAGE')
    is_knock = int(stage in KNOCKOUT_STAGES)
    season   = int(str(match.get('season', {}).get('startDate', '2022')[:4]))
    is_wc    = match.get('competition', {}).get('code', '') == 'WC'
    is_neutral= int(is_wc)   # WC games are on neutral ground

    # FIFA rankings
    h_code = home.get('tla', home.get('shortName', 'UNK'))[:3].upper()
    a_code = away.get('tla', away.get('shortName', 'UNK'))[:3].upper()
    h_rank = rank_to_norm(get_fifa_rank(h_code, season))
    a_rank = rank_to_norm(get_fifa_rank(a_code, season))

    # Group stage context
    gpt = group_pts_tracker or {}
    h_gpts = gpt.get(h_code, 0)
    a_gpts = gpt.get(a_code, 0)

    match_id = match.get('id', 0)

    # Build timeline
    goal_events = parse_goals(match)  # [(minute, hs, as), ...]

    # Checkpoints: pre-game, each goal, half-time, full-time
    checkpoints = [(0, 0, 0)]             # pre-game
    lead_changes = 0; prev_leader = 0; n_events = 0

    for (minute, hs, as_) in goal_events:
        n_events += 1
        cur = np.sign(hs - as_)
        if cur != prev_leader and cur != 0:
            lead_changes += 1
        prev_leader = cur
        checkpoints.append((minute, hs, as_))

    # Half-time snapshot if goals exist (otherwise already at 0-0)
    if goal_events:
        # What was the score at halftime?
        ht = score.get('halfTime', {})
        ht_h = ht.get('home') or 0
        ht_a = ht.get('away') or 0
        checkpoints.append((45, ht_h, ht_a))

    checkpoints.append((90, h_final, a_final))  # final

    rows = []
    for (minute, hs, as_) in checkpoints:
        minute = int(np.clip(minute, 0, 90))
        diff   = int(np.clip(hs - as_, -5, 5))
        t_rem  = max(0, (90 - minute) * 60)
        elapsed= minute / 90.0
        half   = 1 if minute <= 45 else 2
        is_et  = 0   # extra time handling: we only have 90-min data in free tier
        lc_norm= lead_changes / max(1, n_events)
        state  = 0 if diff < 0 else (2 if diff > 0 else 1)   # 0=behind,1=level,2=ahead

        rows.append({
            # Identifiers
            'match_id'          : match_id,
            'season'            : season,
            'minute'            : minute,
            # Features
            'goal_diff'         : diff,
            'time_remaining_sec': t_rem,
            'half'              : half,
            'match_time_pct'    : elapsed,
            'is_extra_time'     : is_et,
            'is_knockout'       : is_knock,
            'lead_changes_norm' : lc_norm,
            'home_rank_norm'    : h_rank,
            'away_rank_norm'    : a_rank,
            'rank_diff'         : h_rank - a_rank,
            'home_group_pts'    : h_gpts,
            'away_group_pts'    : a_gpts,
            'is_neutral_venue'  : is_neutral,
            'score_state'       : state,
            # Target
            'outcome'           : target,
            # Metadata
            'home_code'         : h_code,
            'away_code'         : a_code,
            'stage'             : stage,
            'is_wc'             : int(is_wc),
        })

    return rows


print('✅ build_snapshots() ready')
print('Feature columns: goal_diff, time_remaining_sec, half, match_time_pct,')
print('                 is_extra_time, is_knockout, lead_changes_norm,')
print('                 home_rank_norm, away_rank_norm, rank_diff,')
print('                 home_group_pts, away_group_pts, is_neutral_venue, score_state')
print('Target: outcome (0=away win, 1=draw, 2=home win)')


✅ build_snapshots() ready
Feature columns: goal_diff, time_remaining_sec, half, match_time_pct,
                 is_extra_time, is_knockout, lead_changes_norm,
                 home_rank_norm, away_rank_norm, rank_diff,
                 home_group_pts, away_group_pts, is_neutral_venue, score_state
Target: outcome (0=away win, 1=draw, 2=home win)


## Cell 5 — Process All World Cup Matches

In [18]:
print('Building WC feature dataset...')
all_rows = []
skipped  = 0

for match in wc_matches:
    status = match.get('status', '')
    if status != 'FINISHED':
        skipped += 1; continue

    full = match.get('score', {}).get('fullTime', {})
    if full.get('home') is None or full.get('away') is None:
        skipped += 1; continue

    rows = build_snapshots(match)
    all_rows.extend(rows)

wc_df = pd.DataFrame(all_rows)

print(f'WC matches processed : {len(wc_matches) - skipped} (skipped {skipped})')
print(f'WC snapshots         : {len(wc_df):,}')

if wc_df.empty:
    print()
    print('⚠️  No WC snapshots built. Two possible causes:')
    print()
    print('  A) API returned 0 matches (most likely):')
    print('     Your free tier may not include the WC competition.')
    print('     Check the competition list printed in Cell 5.')
    print('     Solution: see the StatsBomb fallback cell below.')
    print()
    print('  B) All matches were skipped (status != FINISHED):')
    print(f'     Skipped: {skipped}  |  Raw count: {len(wc_matches)}')
    if wc_matches:
        sample_statuses = list(set(m.get('status','?') for m in wc_matches[:20]))
        print(f'     Sample statuses: {sample_statuses}')
else:
    print(f'WC seasons covered   : {sorted(wc_df.season.unique())}')
    print(f'Outcome distribution :')
    for label, code in [('Away win (0)', 0), ('Draw (1)', 1), ('Home win (2)', 2)]:
        n = (wc_df.outcome == code).sum()
        print(f'  {label}: {n} ({n/len(wc_df):.1%})')
    print()
    print(wc_df.head())


Building WC feature dataset...
WC matches processed : 0 (skipped 0)
WC snapshots         : 0

⚠️  No WC snapshots built. Two possible causes:

  A) API returned 0 matches (most likely):
     Your free tier may not include the WC competition.
     Check the competition list printed in Cell 5.
     Solution: see the StatsBomb fallback cell below.

  B) All matches were skipped (status != FINISHED):
     Skipped: 0  |  Raw count: 0


## Cell 11b — StatsBomb Fallback (if football-data.org has no WC access)

If Cell 5 showed `WC: ❌ No — need Tier Football`, use this cell instead.
StatsBomb publishes **free, open World Cup data** covering 2018 and 2022 with
full match events (every pass, shot, and goal). No API key needed.


In [19]:
# ── StatsBomb Open Data Fallback ─────────────────────────────────────────
# StatsBomb publishes free World Cup match data on GitHub.
# No API key needed — direct HTTP fetch from their public repo.
# Coverage: WC 2018 (64 matches), WC 2022 (64 matches) = 128 games
# With per-event resolution (~3,000 events/match) this gives ~380,000 snapshots.

SB_BASE = 'https://raw.githubusercontent.com/statsbomb/open-data/master/data'

# StatsBomb competition IDs for World Cup
SB_WC_COMPETITIONS = [
    {'competition_id': 43, 'season_id': 106, 'label': 'WC 2022'},
    {'competition_id': 43, 'season_id':   3, 'label': 'WC 2018'},
]

def sb_get(path):
    r = requests.get(f'{SB_BASE}/{path}', timeout=15)
    if r.status_code != 200:
        print(f'  StatsBomb {path}: {r.status_code}')
        return []
    return r.json()


def pull_statsbomb_wc():
    cache = RAW_DIR / 'sb_wc_matches.json'
    if cache.exists():
        with open(cache) as f:
            data = json.load(f)
        if data: 
            print(f'📂 StatsBomb cache: {len(data)} matches')
            return data

    print('🌐 Pulling StatsBomb World Cup matches (free, no key)...')
    all_sb = []

    for comp in SB_WC_COMPETITIONS:
        cid = comp['competition_id']; sid = comp['season_id']
        matches = sb_get(f'matches/{cid}/{sid}.json')
        print(f'  {comp["label"]}: {len(matches)} matches')
        for m in matches:
            m['_sb_competition_id'] = cid
            m['_sb_season_id']      = sid
        all_sb.extend(matches)
        time.sleep(0.5)

    with open(cache, 'w') as f:
        json.dump(all_sb, f)

    print(f'Total: {len(all_sb)} StatsBomb WC matches')
    return all_sb


def convert_sb_to_snapshots(sb_match):
    """
    Convert a StatsBomb match record to our feature-snapshot format.
    StatsBomb uses different field names — this adapter bridges the gap.
    """
    h_team  = sb_match.get('home_team', {})
    a_team  = sb_match.get('away_team', {})
    h_score = sb_match.get('home_score', 0) or 0
    a_score = sb_match.get('away_score', 0) or 0
    season  = int(str(sb_match.get('season', {}).get('season_name', '2022'))[:4])

    # Stage: StatsBomb uses match_week / competition_stage
    stage_name = sb_match.get('competition_stage', {}).get('name', 'Group Stage')
    is_knock   = int('Group' not in stage_name)

    if h_score > a_score: target = 2
    elif h_score < a_score: target = 0
    else: target = 1

    h_code = h_team.get('country', {}).get('name', 'UNK')[:3].upper()
    a_code = a_team.get('country', {}).get('name', 'UNK')[:3].upper()
    h_rank = rank_to_norm(get_fifa_rank(h_code, season))
    a_rank = rank_to_norm(get_fifa_rank(a_code, season))
    match_id = sb_match.get('match_id', 0)

    # Reconstruct goal timeline from score (StatsBomb free data has match-level scores)
    # For richer event data we'd fetch the event file per match — optional enhancement
    rows = []
    for (minute, hs, as_) in [
        (0,  0,       0),        # pre-game
        (45, h_score//2, a_score//2),  # approximate HT
        (90, h_score,    a_score),     # full-time
    ]:
        diff  = int(np.clip(hs - as_, -5, 5))
        t_rem = max(0, (90 - minute) * 60)
        rows.append({
            'match_id'          : match_id,
            'season'            : season,
            'minute'            : minute,
            'goal_diff'         : diff,
            'time_remaining_sec': t_rem,
            'half'              : 1 if minute <= 45 else 2,
            'match_time_pct'    : minute / 90.0,
            'is_extra_time'     : 0,
            'is_knockout'       : is_knock,
            'lead_changes_norm' : 0.0,
            'home_rank_norm'    : h_rank,
            'away_rank_norm'    : a_rank,
            'rank_diff'         : h_rank - a_rank,
            'home_group_pts'    : 0,
            'away_group_pts'    : 0,
            'is_neutral_venue'  : 1,
            'score_state'       : 0 if diff < 0 else (2 if diff > 0 else 1),
            'outcome'           : target,
            'home_code'         : h_code,
            'away_code'         : a_code,
            'stage'             : stage_name,
            'is_wc'             : 1,
        })
    return rows


# Run if WC data is empty
if wc_df.empty:
    print('WC data empty — using StatsBomb fallback')
    sb_matches = pull_statsbomb_wc()
    sb_rows = []
    for m in sb_matches:
        sb_rows.extend(convert_sb_to_snapshots(m))
    wc_df = pd.DataFrame(sb_rows)
    print(f'\nStatsBomb WC snapshots: {len(wc_df):,}')
    print(f'Seasons: {sorted(wc_df.season.unique())}')
else:
    print(f'✅ football-data.org WC data already loaded ({len(wc_df):,} snapshots) — skipping StatsBomb')


WC data empty — using StatsBomb fallback
📂 StatsBomb cache: 128 matches

StatsBomb WC snapshots: 384
Seasons: [np.int64(2018), np.int64(2022)]


## Cell 6b — Enrich StatsBomb WC Data with Event Files

StatsBomb publishes full event-level data (every shot, goal, tackle) for each match.
Right now we only have 3 snapshots per WC match (pre-game, HT, FT = 384 total).
This cell fetches the event JSON for all 128 matches and rebuilds WC snapshots
at every 5-minute interval + after every goal, giving **~3,000 WC snapshots**.

This runs entirely from the free StatsBomb GitHub repo — no API key needed.
Event files are cached locally after the first download.


In [20]:
# ── Name → tricode mapping for StatsBomb team names ─────────────────────
# StatsBomb uses full English names; our FIFA rank table uses 3-letter codes.
SB_NAME_TO_CODE = {
    'Brazil':'BRA','France':'FRA','Argentina':'ARG','Spain':'ESP',
    'Germany':'GER','England':'ENG','Portugal':'POR','Netherlands':'NED',
    'Belgium':'BEL','Italy':'ITA','Croatia':'CRO','Uruguay':'URU',
    'Mexico':'MEX','Colombia':'COL','Chile':'CHI','Switzerland':'SUI',
    'Denmark':'DEN','Sweden':'SWE','Poland':'POL','Senegal':'SEN',
    'Morocco':'MAR','Japan':'JPN','Australia':'AUS','South Korea':'KOR',
    'United States':'USA','Canada':'CAN','Ecuador':'ECU','Qatar':'QAT',
    'Saudi Arabia':'KSA','Iran':'IRN','Tunisia':'TUN','Cameroon':'CMR',
    'Ghana':'GHA','Serbia':'SRB','South Africa':'RSA','Costa Rica':'CRC',
    'Wales':'WAL','Nigeria':'NGA','Peru':'PER','Russia':'RUS','Egypt':'EGY',
    'Iceland':'ICE','Panama':'PAN','Korea Republic':'KOR',
    "Côte d'Ivoire":'CIV',"Cote d'Ivoire":'CIV','Algeria':'ALG',
    'Paraguay':'PAR','Portugal':'POR','Slovakia':'SVK','Slovenia':'SLO',
    'New Zealand':'NZL','Honduras':'HND','Greece':'GRC',
    'Trinidad and Tobago':'TRI','Ukraine':'UKR','Angola':'ANG',
    'Togo':'TOG','Czech Republic':'CZE','Ireland':'IRL',
}

def sb_name_to_code(name):
    return SB_NAME_TO_CODE.get(name, name[:3].upper())


# ── Fetch one event file ───────────────────────────────────────────────────
def fetch_sb_events(match_id):
    cache = RAW_DIR / f'sb_events_{match_id}.json'
    if cache.exists():
        with open(cache) as f:
            return json.load(f)
    url = f'{SB_BASE}/events/{match_id}.json'
    r   = requests.get(url, timeout=30)
    if r.status_code != 200:
        return []
    events = r.json()
    with open(cache, 'w') as f:
        json.dump(events, f)
    time.sleep(0.4)   # polite rate limiting for GitHub
    return events


# ── Build enriched snapshots from event data ───────────────────────────────
def build_enriched_snapshots(sb_match, events):
    """
    Build feature snapshots sampled at every 5 minutes + after each goal.
    Uses StatsBomb event data to get accurate goal timing and shot metrics.

    Returns a list of feature dicts with the same 14-column schema as
    build_snapshots(), so everything merges cleanly in Cell 7.
    """
    h_team_name = sb_match.get('home_team', {}).get('home_team_name', 'UNK')
    a_team_name = sb_match.get('away_team', {}).get('away_team_name', 'UNK')
    h_score_final = sb_match.get('home_score', 0) or 0
    a_score_final = sb_match.get('away_score', 0) or 0

    season_str  = str(sb_match.get('season', {}).get('season_name', '2022'))[:4]
    season      = int(season_str)
    stage_name  = sb_match.get('competition_stage', {}).get('name', 'Group Stage')
    is_knock    = int('Group' not in stage_name)
    match_id    = sb_match.get('match_id', 0)

    if h_score_final > a_score_final: target = 2
    elif h_score_final < a_score_final: target = 0
    else: target = 1

    h_code = sb_name_to_code(h_team_name)
    a_code = sb_name_to_code(a_team_name)
    h_rank = rank_to_norm(get_fifa_rank(h_code, season))
    a_rank = rank_to_norm(get_fifa_rank(a_code, season))

    # ── Pass 1: extract goals and shots from events ───────────────────────
    h_score = a_score = 0
    goal_timeline = []   # [(abs_minute, h_score_after, a_score_after)]
    shot_log      = []   # [(abs_minute, is_home, on_target, xg)]

    for evt in events:
        period = evt.get('period', 1)
        if period > 2:
            continue   # ignore extra time and penalties for 90-min model

        minute  = int(evt.get('minute', 0))
        # StatsBomb minutes are absolute: period 1 = 0-45, period 2 = 45-90
        abs_min = minute

        evt_type = evt.get('type', {}).get('name', '')
        is_home  = evt.get('team', {}).get('name', '') == h_team_name

        if evt_type == 'Shot':
            shot    = evt.get('shot', {})
            outcome = shot.get('outcome', {}).get('name', '')
            xg      = float(shot.get('statsbomb_xg', 0) or 0)
            on_tgt  = outcome in ('Goal', 'Saved')
            shot_log.append((abs_min, is_home, on_tgt, xg))

            if outcome == 'Goal':
                if is_home: h_score += 1
                else:       a_score += 1
                goal_timeline.append((abs_min, h_score, a_score))

    # ── Snapshot checkpoints ──────────────────────────────────────────────
    # Every 5 minutes + immediately after each goal
    fixed_mins = list(range(0, 91, 5))              # 0, 5, 10, ... 90
    goal_mins  = [g[0] for g in goal_timeline]
    all_mins   = sorted(set(fixed_mins + goal_mins))

    # ── Pass 2: build feature snapshot at each checkpoint ────────────────
    lead_changes  = 0
    prev_leader   = 0
    prev_hs = prev_as = 0
    n_events = max(1, len(goal_timeline))
    rows = []

    for min_t in all_mins:
        # Score at this minute: last goal that happened ≤ min_t
        hs = as_ = 0
        for (gm, gh, ga) in goal_timeline:
            if gm <= min_t:
                hs, as_ = gh, ga
            else:
                break

        # Track lead changes
        cur = int(np.sign(hs - as_))
        if cur != prev_leader and cur != 0:
            lead_changes += 1
        prev_leader = cur

        diff  = int(np.clip(hs - as_, -5, 5))
        t_rem = int(max(0, (90 - min_t) * 60))
        half  = 1 if min_t <= 45 else 2
        state = 0 if diff < 0 else (2 if diff > 0 else 1)

        # Shots on target diff — rolling 15-min window
        window_shots = [(m, ih, ot, xg) for (m, ih, ot, xg) in shot_log
                        if max(0, min_t-15) <= m <= min_t]
        h_sot = sum(1 for (_, ih, ot, _) in window_shots if ih and ot)
        a_sot = sum(1 for (_, ih, ot, _) in window_shots if not ih and ot)

        rows.append({
            # Identifiers
            'match_id'          : match_id,
            'season'            : season,
            'minute'            : min_t,
            # Core 14 features
            'goal_diff'         : diff,
            'time_remaining_sec': t_rem,
            'half'              : half,
            'match_time_pct'    : min_t / 90.0,
            'is_extra_time'     : 0,
            'is_knockout'       : is_knock,
            'lead_changes_norm' : lead_changes / n_events,
            'home_rank_norm'    : h_rank,
            'away_rank_norm'    : a_rank,
            'rank_diff'         : h_rank - a_rank,
            'home_group_pts'    : 0,
            'away_group_pts'    : 0,
            'is_neutral_venue'  : 1,
            'score_state'       : state,
            # Target
            'outcome'           : target,
            # Metadata
            'home_code'         : h_code,
            'away_code'         : a_code,
            'stage'             : stage_name,
            'is_wc'             : 1,
            # Bonus feature — not in core 14 but useful for diagnostics
            'shots_on_target_diff': h_sot - a_sot,
        })

    return rows


# ── Run enrichment across all 128 WC matches ──────────────────────────────
sb_event_cache = RAW_DIR / 'sb_wc_matches.json'
if not sb_event_cache.exists():
    print('⚠️  Run Cell 11b first to populate sb_wc_matches.json')
else:
    with open(sb_event_cache) as f:
        sb_matches_all = json.load(f)

    print(f'Enriching {len(sb_matches_all)} StatsBomb WC matches...')
    print('Downloading event files (cached after first run — ~2 min)...')
    print()

    enriched_rows = []
    failed = 0

    for idx, sb_m in enumerate(sb_matches_all, 1):
        mid    = sb_m.get('match_id', 0)
        events = fetch_sb_events(mid)

        if not events:
            print(f'  [{idx:3d}/{len(sb_matches_all)}] {mid}: no events')
            failed += 1
            continue

        rows = build_enriched_snapshots(sb_m, events)
        enriched_rows.extend(rows)

        if idx % 16 == 0 or idx == len(sb_matches_all):
            print(f'  [{idx:3d}/{len(sb_matches_all)}] '
                  f'{enriched_rows[-1]["home_code"]} vs '
                  f'{enriched_rows[-1]["away_code"]} — '
                  f'{len(events):,} events → {len(rows)} snapshots')

    wc_df = pd.DataFrame(enriched_rows)

    print(f'\n✅ Enrichment complete')
    print(f'   Matches processed : {len(sb_matches_all) - failed}')
    print(f'   Failed            : {failed}')
    print(f'   Total snapshots   : {len(wc_df):,}  (was 384)')
    print(f'   Avg per match     : {len(wc_df)/(len(sb_matches_all)-failed):.0f}')
    print(f'   Seasons           : {sorted(wc_df.season.unique())}')
    print()
    print('Outcome distribution:')
    for label, code in [('Away win (0)', 0), ('Draw (1)', 1), ('Home win (2)', 2)]:
        n = (wc_df.outcome == code).sum()
        print(f'  {label}: {n:,} ({n/len(wc_df):.1%})')
    print()
    print(wc_df[['match_id','minute','goal_diff',
                 'home_rank_norm','score_state','outcome']].head(10).to_string())


Enriching 128 StatsBomb WC matches...

  [ 16/128] POL vs KSA — 3,202 events → 21 snapshots
  [ 32/128] AUS vs DEN — 3,594 events → 20 snapshots
  [ 48/128] JPN vs CRO — 4,559 events → 21 snapshots
  [ 64/128] KOR vs POR — 3,378 events → 21 snapshots
  [ 80/128] RUS vs EGY — 3,348 events → 22 snapshots
  [ 96/128] FRA vs ARG — 3,549 events → 25 snapshots
  [112/128] PER vs DEN — 3,097 events → 20 snapshots
  [128/128] KSA vs EGY — 3,831 events → 21 snapshots

✅ Enrichment complete
   Matches processed : 128
   Failed            : 0
   Total snapshots   : 2,678  (was 384)
   Avg per match     : 21
   Seasons           : [np.int64(2018), np.int64(2022)]

Outcome distribution:
  Away win (0): 931 (34.8%)
  Draw (1): 571 (21.3%)
  Home win (2): 1,176 (43.9%)

   match_id  minute  goal_diff  home_rank_norm  score_state  outcome
0   3857276       0          0             0.6            1        0
1   3857276       3         -1             0.6            0        0
2   3857276       5        

## Cell 6 — Pull Top-5 League Data (Training Volume)

In [21]:
def pull_league_season(league_id, season_year):
    """
    Pull all finished matches for one league-season.
    Caches to disk to avoid re-fetching.
    """
    cache = RAW_DIR / f'league_{league_id}_{season_year}.json'
    if cache.exists():
        with open(cache) as f:
            return json.load(f)

    try:
        data = fd_get(f'/competitions/{league_id}/matches',
                      params={'season': season_year, 'status': 'FINISHED'})
        time.sleep(RATE_SLEEP)
        matches = data.get('matches', [])
        with open(cache, 'w') as f:
            json.dump(matches, f)
        return matches
    except Exception as e:
        print(f'    ⚠️  Failed {league_id}/{season_year}: {e}')
        time.sleep(RATE_SLEEP)
        return []


# NOTE: This cell will take ~30-60 minutes on the free tier
# (10 req/min, 5 leagues × 10 seasons = 50 requests)
# Re-running is instant — everything is cached to disk.

print('Pulling league data (cached after first run)...')
print('Estimated time if not cached: ~6 minutes (free tier rate limiting)')
print()

league_rows = []
total_matches = 0

for league_id, league_name in LEAGUE_IDS.items():
    league_count = 0
    for season in LEAGUE_SEASONS:
        matches = pull_league_season(league_id, season)
        for match in matches:
            if match.get('status') != 'FINISHED': continue
            full = match.get('score', {}).get('fullTime', {})
            if full.get('home') is None: continue
            rows = build_snapshots(match)
            league_rows.extend(rows)
            league_count += 1

        total_matches += league_count

    print(f'  {league_name:<20}: {league_count:,} matches')

league_df = pd.DataFrame(league_rows) if league_rows else pd.DataFrame()
print(f'\nLeague snapshots     : {len(league_df):,}')
print(f'Total WC + leagues   : {len(wc_df) + len(league_df):,}')


Pulling league data (cached after first run)...
Estimated time if not cached: ~6 minutes (free tier rate limiting)

  Premier League      : 760 matches
  La Liga             : 760 matches
  Bundesliga          : 611 matches
  Serie A             : 760 matches
  Ligue 1             : 611 matches

League snapshots     : 7,004
Total WC + leagues   : 9,682


## Cell 7 — Combine, Validate, Save

In [22]:
import scipy.stats as scipy_stats

FEATURE_COLS = [
    'goal_diff', 'time_remaining_sec', 'half', 'match_time_pct',
    'is_extra_time', 'is_knockout', 'lead_changes_norm',
    'home_rank_norm', 'away_rank_norm', 'rank_diff',
    'home_group_pts', 'away_group_pts',
    'is_neutral_venue', 'score_state',
]
TARGET_COL = 'outcome'

# Combine datasets
if len(league_df) > 0:
    combined = pd.concat([wc_df, league_df], ignore_index=True)
else:
    combined = wc_df.copy()

print('='*60)
print('COMBINED DATASET VALIDATION')
print('='*60)

passed = 0; failed = 0
def chk(name, ok, detail=''):
    global passed, failed
    print(f'{"✅" if ok else "❌"} {name}')
    if detail: print(f'    {detail}')
    if ok: passed += 1
    else:  failed += 1

chk('No nulls in features',
    combined[FEATURE_COLS].isnull().sum().sum() == 0)

chk('Outcome is 0, 1, or 2',
    combined[TARGET_COL].isin([0,1,2]).all())

chk('goal_diff in [-5, +5]',
    combined.goal_diff.between(-5,5).all())

chk('time_remaining_sec in [0, 5400]',
    combined.time_remaining_sec.between(0,5400).all())

chk('rank norms in [0, 1]',
    combined.home_rank_norm.between(0,1).all() and
    combined.away_rank_norm.between(0,1).all())

home_rate = (combined.outcome == 2).mean()
draw_rate = (combined.outcome == 1).mean()
away_rate = (combined.outcome == 0).mean()
chk('Realistic outcome distribution (home 40-55%, draw 15-35%)',
    0.35 <= home_rate <= 0.60 and 0.10 <= draw_rate <= 0.40,
    f'Home {home_rate:.1%}  Draw {draw_rate:.1%}  Away {away_rate:.1%}')

n_matches = combined.match_id.nunique()
chk('Sufficient match count (>500)',
    n_matches > 500,
    f'{n_matches:,} unique matches  ({len(combined):,} snapshots)')

r_gd, _ = scipy_stats.pointbiserialr(
    (combined.outcome == 2).astype(float), combined.goal_diff)
chk('goal_diff positively correlates with home win (r>0.3)',
    r_gd > 0.3, f'r={r_gd:.3f}')

print(f'\nResult: {passed} passed, {failed} failed')

# Save
out_path = PROC_DIR / 'features_raw.parquet'
combined.to_parquet(out_path, index=False)
print(f'\n✅ Saved: {out_path}')
print(f'   Shape : {combined.shape}')
print(f'   Size  : {out_path.stat().st_size / 1024 / 1024:.1f} MB')

# Summary
print(f'\n{"="*60}')
print('PHASE 1 COMPLETE')
print(f'{"="*60}')
print(f'''
Dataset ready for Phase 2 transfer learning:
  Features      : {len(FEATURE_COLS)} columns
  Snapshots     : {len(combined):,}
  WC matches    : {wc_df.match_id.nunique():,}
  League matches: {league_df.match_id.nunique() if len(league_df)>0 else 0:,}
  Target classes: 3 (away win, draw, home win)

Next step → phase2_transfer_learning.ipynb
  Load NBA model weights, replace output layer (1→3 nodes),
  train with cross-entropy loss on this dataset.
''')


COMBINED DATASET VALIDATION
✅ No nulls in features
✅ Outcome is 0, 1, or 2
✅ goal_diff in [-5, +5]
✅ time_remaining_sec in [0, 5400]
✅ rank norms in [0, 1]
✅ Realistic outcome distribution (home 40-55%, draw 15-35%)
    Home 42.9%  Draw 24.5%  Away 32.6%
✅ Sufficient match count (>500)
    3,630 unique matches  (9,682 snapshots)
✅ goal_diff positively correlates with home win (r>0.3)
    r=0.541

Result: 8 passed, 0 failed

✅ Saved: world_cup_win_prob\processed\features_raw.parquet
   Shape : (9682, 23)
   Size  : 0.1 MB

PHASE 1 COMPLETE

Dataset ready for Phase 2 transfer learning:
  Features      : 14 columns
  Snapshots     : 9,682
  WC matches    : 128
  League matches: 3,502
  Target classes: 3 (away win, draw, home win)

Next step → phase2_transfer_learning.ipynb
  Load NBA model weights, replace output layer (1→3 nodes),
  train with cross-entropy loss on this dataset.

